## 1. Introduction <a id='s1'></a>

### 1.1 Motivation

Imagine standing at **Rila Monastery** planning your route to the **Seven Rila Lakes**.
Multiple paths exist — but which one is truly *optimal*?

A naive answer: *the shortest one*. But in mountain terrain this is rarely correct.
A shorter path with steep elevation gain can take **twice as long** and be far more exhausting
than a longer path with gentle slopes.

> **Key Question:** *Is the shortest path always the best path in the mountains?*

### 1.2 Real-World Applications

-  **Trail running & hiking** — plan routes by effort, not just distance
-  **Mountain rescue** — find the fastest safe route to a location
-  **Navigation apps** — Komoot, Wikiloc need elevation-aware routing

### 1.3 The Two Tracks

| Track | Mountain | Distance | Elev. Gain | Max Alt. |
|---|---|---|---|---|
| Rila Monastery → Seven Lakes | Rila | 37.47 km | +2,947 m | 2,735 m |
| Vihren Peak | Pirin | 8.87 km | +1,053 m | 2,923 m |

### 1.4 Three Definitions of 'Optimal'

-  **Shortest distance** — minimize total kilometers
-  **Least elevation gain** — minimize total ascent
-  **Fastest time** — minimize hiking time via Naismith's Rule

---

## 2. Mathematical Background <a id='s2'></a>

### 2.1 Graph Theory

A **weighted graph** is defined as $G = (V, E, w)$ where:
- $V$ = vertices (GPS points)
- $E$ = edges (connections between consecutive points)
- $w: E \rightarrow \mathbb{R}^+$ = weight function

### 2.2 Haversine Distance

GPS coordinates lie on a sphere — we use the **Haversine formula**:

$$d = 2R \cdot \arctan\!\left(\sqrt{\frac{a}{1-a}}\right), \quad a = \sin^2\!\frac{\Delta\phi}{2} + \cos\phi_1\cos\phi_2\sin^2\!\frac{\Delta\lambda}{2}$$

### 2.3 Naismith's Rule (1892)

$$T = \frac{d}{5} + \frac{h}{600}$$

Where $T$ = time (hours), $d$ = distance (km), $h$ = elevation gain (m).

Per edge: $w_t(u,v) = \dfrac{d(u,v)}{5000} + \dfrac{\max(0,\, e_v - e_u)}{600}$

### 2.4 Dijkstra's Algorithm (1959)

Finds minimum-cost path from source $s$:
$$\delta(s,v) = \min_{p:\,s\rightsquigarrow v}\sum_{e\in p} w(e)$$

**Relaxation step** — the core of the algorithm:
$$\text{if } dist[u] + w(u,v) < dist[v] \Rightarrow dist[v] \leftarrow dist[u] + w(u,v)$$

**Complexity:** $O\bigl((|V|+|E|)\log|V|\bigr)$ with a min-heap.

All three weight functions satisfy the non-negativity requirement ✅

---

## 3. Data Loading & Graph Construction <a id='s3'></a>

We parse GPX files using Python's built-in `xml.etree.ElementTree` and convert
each track into a weighted directed graph — no external libraries needed.

---

In [ ]:
import xml.etree.ElementTree as ET
import math, heapq, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Haversine distance ──────────────────────────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6_371_000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi    = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

# ── GPX parser ─────────────────────────────────────────────
def parse_gpx(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    ns   = {'gpx': 'http://www.topografix.com/GPX/1/1'}
    pts  = []
    for trkpt in root.findall('.//gpx:trkpt', ns):
        lat = float(trkpt.get('lat'))
        lon = float(trkpt.get('lon'))
        ele = trkpt.find('gpx:ele', ns)
        pts.append((lat, lon, float(ele.text) if ele is not None else 0.0))
    return pts

# ── Graph builder ──────────────────────────────────────────
def build_graph(points):
    graph = {i: [] for i in range(len(points))}
    for i in range(len(points)-1):
        lat1,lon1,e1 = points[i]
        lat2,lon2,e2 = points[i+1]
        d  = haversine(lat1,lon1,lat2,lon2)
        eg = max(0.0, e2-e1)
        eb = max(0.0, e1-e2)
        nt = (d/1000)/5 + eg/600
        nb = (d/1000)/5 + eb/600
        graph[i  ].append((i+1, d, eg, nt))
        graph[i+1].append((i,   d, eb, nb))
    return graph

# ── Load data ──────────────────────────────────────────────
RILA_PATH   = '../data/bulgaria-rila-mountains-2-day-hike-rila-monastery-seven-lake.gpx'
VIHREN_PATH = '../data/Vihren-Pirin'

rila_pts   = parse_gpx(RILA_PATH)
vihren_pts = parse_gpx(VIHREN_PATH)
rila_g     = build_graph(rila_pts)
vihren_g   = build_graph(vihren_pts)

print(f'Rila   : {len(rila_pts):,} points loaded')
print(f'Vihren : {len(vihren_pts):,} points loaded')

In [ ]:
# ── Elevation profiles ─────────────────────────────────────
def cum_dist(pts):
    d = [0.0]
    for i in range(1, len(pts)):
        d.append(d[-1] + haversine(*pts[i-1][:2], *pts[i][:2]) / 1000)
    return d

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
fig.suptitle('Elevation Profiles — Real Bulgarian Mountain GPS Tracks',
             fontsize=14, fontweight='bold')

for pts, title, color, ax in [
    (rila_pts,   'Rila: Monastery → Seven Lakes', '#2ECC71', axes[0]),
    (vihren_pts, 'Vihren Peak — Pirin',           '#E74C3C', axes[1]),
]:
    ds = cum_dist(pts)
    es = [p[2] for p in pts]
    ax.fill_between(ds, es, min(es), alpha=0.2, color=color)
    ax.plot(ds, es, color=color, linewidth=1.2)
    mi = es.index(max(es))
    ax.scatter([ds[0]],  [es[0]],  s=80, color='green', zorder=5, label='Start')
    ax.scatter([ds[-1]], [es[-1]], s=80, color='red',   zorder=5, label='End')
    ax.scatter([ds[mi]], [es[mi]], s=100,color='gold',  zorder=5, label=f'Peak {max(es):.0f}m')
    ax.set_title(title); ax.set_xlabel('Distance (km)'); ax.set_ylabel('Elevation (m)')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Dijkstra's Algorithm — From Scratch <a id='s4'></a>

---

In [ ]:
# ── Dijkstra ───────────────────────────────────────────────
def dijkstra(graph, source, weight_index):
    """
    Dijkstra shortest path with min-heap.
    weight_index: 1=distance, 2=elevation, 3=naismith
    """
    dist    = {n: math.inf for n in graph}
    prev    = {n: None     for n in graph}
    dist[source] = 0.0
    pq      = [(0.0, source)]
    visited = set()

    while pq:
        cost, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        for edge in graph[u]:
            v, w = edge[0], edge[weight_index]
            cand = cost + w
            if cand < dist[v]:         # ← relaxation step
                dist[v] = cand
                prev[v] = u
                heapq.heappush(pq, (cand, v))

    return dist, prev


def reconstruct_path(prev, source, target):
    path, node = [], target
    while node is not None:
        path.append(node); node = prev[node]
    path.reverse()
    return path if path[0] == source else []


# ── Run all combinations ───────────────────────────────────
configs = [
    (1, 'Shortest distance',  1/1000, 'km'),
    (2, 'Least elev. gain',   1,      'm' ),
    (3, 'Fastest (Naismith)', 1,      'h' ),
]

results = {}
for track_name, graph, pts in [('Rila', rila_g, rila_pts), ('Vihren', vihren_g, vihren_pts)]:
    target = len(pts) - 1
    results[track_name] = {}
    print(f'\n{track_name}')
    print('-' * 40)
    for wi, label, scale, unit in configs:
        t0 = time.time()
        dm, pm = dijkstra(graph, 0, wi)
        ms = (time.time()-t0)*1000
        path = reconstruct_path(pm, 0, target)
        cost = dm[target]
        results[track_name][label] = {'path': path, 'cost': cost, 'pts': pts}
        print(f'  {label:25s}: {cost*scale:8.2f} {unit}  ({ms:.1f} ms)')

## 5. Route Comparison & Visualizations <a id='s5'></a>

---

In [ ]:
# ── Helper: path stats ─────────────────────────────────────
def path_stats(path, pts):
    d, eg = 0.0, 0.0
    for i in range(len(path)-1):
        a, b = pts[path[i]], pts[path[i+1]]
        d  += haversine(a[0],a[1],b[0],b[1])
        eg += max(0, b[2]-a[2])
    return d/1000, eg, d/5000 + eg/600


# ── Figure 1: Grouped bar chart ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Route Comparison — Three Optimization Criteria',
             fontsize=14, fontweight='bold')

metrics   = ['Distance (km)', 'Elev. Gain (m)', 'Naismith Time (h)']
colors    = ['#3498DB', '#E74C3C', '#F39C12']
opt_labels= ['Shortest distance', 'Least elev. gain', 'Fastest (Naismith)']

for ti, (track_name, graph, pts) in enumerate([
        ('Rila', rila_g, rila_pts),
        ('Vihren', vihren_g, vihren_pts)]):

    # Compute stats for every optimized path
    all_stats = []
    for wi, label, *_ in configs:
        path = results[track_name][label]['path']
        all_stats.append(path_stats(path, pts))
    # all_stats[i] = (dist_km, elev_m, naismith_h) for optimization i

    x = np.arange(len(opt_labels))
    for mi, (metric, ax) in enumerate(zip(metrics, axes)):
        vals = [s[mi] for s in all_stats]
        bars = ax.bar(x + ti*0.35, vals, width=0.32,
                      label=track_name, color=colors[ti], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3*max(vals)/20,
                    f'{v:.1f}', ha='center', va='bottom', fontsize=8)
        ax.set_xticks(x+0.175)
        ax.set_xticklabels(opt_labels, rotation=12, ha='right', fontsize=8)
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: Elevation profiles of the 3 routes ──────────
for track_name, pts in [('Rila', rila_pts), ('Vihren', vihren_pts)]:
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.set_title(f'{track_name} — Elevation Profile by Optimization Strategy',
                 fontsize=12, fontweight='bold')

    line_styles = ['-', '--', ':']
    line_colors = ['#3498DB', '#E74C3C', '#F39C12']

    for (wi, label, *_), ls, lc in zip(configs, line_styles, line_colors):
        path = results[track_name][label]['path']
        path_pts = [pts[i] for i in path]
        ds = cum_dist(path_pts)
        es = [p[2] for p in path_pts]
        ax.plot(ds, es, linestyle=ls, color=lc, linewidth=1.8, label=label)

    ax.set_xlabel('Distance (km)')
    ax.set_ylabel('Elevation (m)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Figure 3: Naismith breakdown ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Naismith Time Breakdown — Distance vs Elevation Component",
             fontsize=13, fontweight='bold')

for ax, (track_name, pts) in zip(axes, [('Rila', rila_pts), ('Vihren', vihren_pts)]):
    dist_comps, elev_comps, labels_bar = [], [], []
    for wi, label, *_ in configs:
        path = results[track_name][label]['path']
        dk, eg, nt = path_stats(path, pts)
        dist_comps.append(dk/5)
        elev_comps.append(eg/600)
        labels_bar.append(label.replace(' ', '\n'))

    x = np.arange(len(labels_bar))
    ax.bar(x, dist_comps, label='Distance (d/5)', color='#3498DB', alpha=0.85)
    ax.bar(x, elev_comps, bottom=dist_comps, label='Elevation (h/600)', color='#E74C3C', alpha=0.85)
    for i,(d,e) in enumerate(zip(dist_comps,elev_comps)):
        ax.text(i, d+e+0.05, f'{d+e:.2f}h', ha='center', fontweight='bold', fontsize=10)
    ax.set_xticks(x); ax.set_xticklabels(labels_bar, fontsize=9)
    ax.set_ylabel('Time (hours)'); ax.set_title(track_name)
    ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ──────────────────────────────────────────
print('\n' + '='*70)
print('FULL COMPARISON TABLE')
print('='*70)
header = f'{"Track":<10} {"Optimized for":<22} {"Dist(km)":>10} {"Gain(m)":>10} {"Time(h)":>10}'
print(header)
print('-'*70)

for track_name, pts in [('Rila', rila_pts), ('Vihren', vihren_pts)]:
    for wi, label, *_ in configs:
        path = results[track_name][label]['path']
        dk, eg, nt = path_stats(path, pts)
        print(f'{track_name:<10} {label:<22} {dk:>10.2f} {eg:>10.0f} {nt:>10.2f}')
    print('-'*70)

## 6. Conclusion <a id='s6'></a>

### 6.1 Answering the Key Question

> *Is the shortest path always the best path in the mountains?*

**No.** The results clearly show that optimizing for different criteria produces
meaningfully different routes — and the 'best' path depends entirely on your goal.

For example, on the Rila track:
- The **shortest distance** path minimizes kilometers but may require steep climbs
- The **least elevation** path is gentler but potentially longer
- The **Naismith-optimal** path balances both — and is the most practically useful






---